# P1-27b · Experiment E3, data: mini-MIAS with verified expert masks (5 folds by woman) and reader masks of the hospital test images (CPU, ~15 min)
The public UEF/VinDr-Mammo record contains only the masks (the images must be requested from PhysioNet under a data-use agreement), so
E3 trains on mini-MIAS (322 digitized screen-film mammograms, 161 women) with the expert-corrected breast and pectoral masks of
Verboom et al. in the orientation verified visually in P1-26b. Writes only `MyDrive/P1_v2/e3m/` and `MyDrive/P1_v2/e3_refs/`.

In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, re, io, shutil
import numpy as np, pandas as pd, cv2
ROOT = '/content/drive/MyDrive/QSEG_FINAL'          # read only
OUT  = '/content/drive/MyDrive/P1_v2'; MIAS = f'{OUT}/mias_ext'; RC = f'{OUT}/revision_checks'
E3   = f'{OUT}/e3m'; REFS = f'{OUT}/e3_refs'
for d in (f'{E3}/data/images', f'{E3}/data/masks', REFS): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{RC}/mias_orientation_final.csv'), 'run P1-26b first'
sys.path.insert(0, ROOT + '/code')
from qseg.config import Config
from qseg.data_prep import chest_wall_side, pad_to_square, generate_breast_mask, canonical_id
print('ok')

In [ ]:
# 2 · mini-MIAS with the verified expert masks (breast-plus-pectoral foreground, labels 1 and 2) and five folds by woman
mm = pd.read_csv(f'{MIAS}/manifest.csv'); mm = mm[mm.status == 'ok'].drop_duplicates('file_id').reset_index(drop=True)
rows = []
for f in mm.file_id:
    shutil.copy(f'{MIAS}/images/{f}.png', f'{E3}/data/images/{f}.png')
    shutil.copy(f'{RC}/masks_final/full/{f}.png', f'{E3}/data/masks/{f}.png')
    m = cv2.imread(f'{E3}/data/masks/{f}.png', 0) > 127
    rows.append(dict(file_id=f, patient_id=f'w{(int(f[3:]) + 1) // 2:03d}', mask_fraction_256=float(m.mean())))
man = pd.DataFrame(rows)
# five folds by woman: women sorted, shuffled once with numpy RandomState(42), dealt round-robin to folds 0-4;
# for fold k: test = fold k, validation = fold (k + 1) mod 5, training = the other three folds (as in Paper 2)
groups = sorted(man.patient_id.unique()); np.random.RandomState(42).shuffle(groups)
fold_of = {g: i % 5 for i, g in enumerate(groups)}; man['fold'] = man.patient_id.map(fold_of)
for k in range(5):
    man[f'split{k}'] = np.where(man.fold == k, 'test', np.where(man.fold == (k + 1) % 5, 'val', 'train'))
man['split'] = man['split0']
man.to_csv(f'{E3}/data/manifest.csv', index=False)
print('images:', len(man), '| women:', man.patient_id.nunique(), '| images per fold:', man.fold.value_counts().sort_index().to_dict())
print('women split across folds:', int((man.groupby('patient_id').fold.nunique() > 1).sum()))
print(man.groupby('fold').mask_fraction_256.describe().round(3).to_string())

In [ ]:
# 6 · Reader 1 masks of the 84 hospital test images at 256 x 256 (rasterization as notebook P1-23)
mn = pd.read_csv(ROOT + '/results/data/manifest.csv'); test = mn[mn.split == 'test'].reset_index(drop=True)
raw = {}
for f in test.original_filename:
    hits = [h for h in glob.glob(f'{ROOT}/**/{glob.escape(f)}', recursive=True) if '/results/' not in h]
    raw[f] = hits[0] if hits else None
assert all(raw.values())
def raster(j, shape):
    H, W = shape; m = {'breast': np.zeros((H, W), np.uint8), 'pectoral': np.zeros((H, W), np.uint8)}
    for s in j['shapes']:
        lab = s['label'].strip().lower()
        if lab in m and s.get('shape_type', 'polygon') == 'polygon': cv2.fillPoly(m[lab], [np.round(np.array(s['points'])).astype(np.int32)], 1)
    return m
def to256(mask, side):
    return (cv2.resize(pad_to_square(mask.astype(np.uint8) * 255, side), (256, 256), interpolation=cv2.INTER_NEAREST) > 127)
J1 = {canonical_id(os.path.basename(p)): json.load(open(p)) for p in glob.glob(f'{OUT}/annot_test/reader1/*.json')}
for rd in ('reader1', 'reader2'):
    for k in ('full', 'breast'): os.makedirs(f'{REFS}/{rd}/{k}', exist_ok=True)
n1 = 0
for _, row in test.iterrows():
    gray = cv2.imread(raw[row.original_filename], cv2.IMREAD_GRAYSCALE)
    proxy, _, _ = generate_breast_mask(gray, 20); side = 'left' if chest_wall_side(proxy) == 'right' else 'right'
    m = raster(J1[row.file_id], gray.shape)
    full = (m['breast'] | m['pectoral']) > 0; br = (m['breast'] > 0) & ~(m['pectoral'] > 0)
    cv2.imwrite(f'{REFS}/reader1/full/{row.file_id}.png', to256(full, side).astype(np.uint8) * 255)
    cv2.imwrite(f'{REFS}/reader1/breast/{row.file_id}.png', to256(br, side).astype(np.uint8) * 255); n1 += 1
print('reader 1 masks written:', n1)

In [ ]:
# 7 · Reader 2 masks (20 images) with the conversion of notebook P1-25 (unchanged)
MARKED = f'{OUT}/annot_test/reader2_marked'
SECOND = pd.read_csv(f'{OUT}/annot_test/reader2_selection.csv').original_filename.tolist()
man = pd.read_csv(ROOT + '/results/data/manifest.csv'); test = man[man.split == 'test'].reset_index(drop=True)
sub = test[test.original_filename.isin(SECOND)].reset_index(drop=True)
print('second-reader images in the test manifest:', len(sub), 'of', len(SECOND)); assert len(sub) == 20
raw = {}
for f in sub.original_filename:
    hits = [h for h in glob.glob(f'{ROOT}/**/{glob.escape(f)}', recursive=True) if '/results/' not in h]
    raw[f] = hits[0] if hits else None
assert all(raw.values()), [k for k, v in raw.items() if v is None]
files = sorted(f for f in glob.glob(f'{MARKED}/*') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))
print('marked files:', len(files)); assert len(files) == 20, 'expected 20 marked images in ' + MARKED

In [ ]:
# 1 · Match every marked image to its original by content (strokes removed), check one-to-one and aspect ratio
def color_masks(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV); h, s, v = hsv[..., 0].astype(int), hsv[..., 1].astype(int), hsv[..., 2].astype(int)
    red = ((h <= 10) | (h >= 165)) & (s >= 90) & (v >= 70)
    yel = (h >= 18) & (h <= 40) & (s >= 90) & (v >= 90)
    return red, yel
def thumb(gray, mask=None, n=96):
    g = gray.astype(np.float32)
    if mask is not None and mask.any(): g[mask] = cv2.medianBlur(gray, 15)[mask]
    t = cv2.resize(g, (n, n), interpolation=cv2.INTER_AREA); t -= t.mean(); return t / (t.std() + 1e-6)
ORIG = {f: cv2.imread(raw[f], cv2.IMREAD_GRAYSCALE) for f in sub.original_filename}
OT = {f: thumb(g) for f, g in ORIG.items()}
rows, MK = [], {}
for p in files:
    bgr = cv2.imread(p); red, yel = color_masks(bgr); g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    t = thumb(g, red | yel); sc = {f: float((t * o).mean()) for f, o in OT.items()}; best = sorted(sc, key=sc.get, reverse=True)
    H, W = ORIG[best[0]].shape
    rows.append(dict(marked=os.path.basename(p), original=best[0], match_corr=sc[best[0]], second_best=sc[best[1]],
                     aspect_marked=bgr.shape[1] / bgr.shape[0], aspect_orig=W / H, red_px=int(red.sum()), yellow_px=int(yel.sum())))
    MK[best[0]] = (bgr, red, yel)
M = pd.DataFrame(rows); M['aspect_diff'] = (M.aspect_marked / M.aspect_orig - 1).abs()
pd.set_option('display.width', 220); print(M.round(3).to_string())
assert M.original.is_unique, 'two marked images matched the same original - check the table'
M['name_agrees'] = [os.path.splitext(a)[0].strip() == os.path.splitext(b)[0].strip() for a, b in zip(M.marked, M.original)]
M['confirmed'] = (M.match_corr > 0.95) & (M.name_agrees | (M.match_corr - M.second_best > 0.03))
print('content match agrees with file name:', int(M.name_agrees.sum()), 'of', len(M))
assert M.confirmed.all(), 'uncertain match - check the table'
print('max aspect difference:', round(M.aspect_diff.max(), 4), '(> 0.02 means the phone cropped the image)')
# (match table already stored by P1-25)

In [ ]:
# 2 · Strokes -> regions on the full-resolution original
def regions(fname):
    gray = ORIG[fname]; H, W = gray.shape; bgr, red, yel = MK[fname]
    red = cv2.resize(red.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
    yel = cv2.resize(yel.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
    proxy, _, _ = generate_breast_mask(gray, 20)
    cw = chest_wall_side(proxy)                      # side of the image where the chest wall is
    def row_boundary(line, gap_to_edge=0.03):
        # per image row: the extreme stroke pixel away from the chest wall; rows between strokes interpolated (robust to gaps);
        # rows beyond the stroke's ends filled only when the stroke ends close to the top/bottom edge
        ys, xs = np.nonzero(line)
        if ys.size < 20: return None
        rows = np.unique(ys)
        ext = np.array([xs[ys == y].max() if cw == 'left' else xs[ys == y].min() for y in rows], float)
        y0, y1 = rows.min(), rows.max()
        if y0 <= gap_to_edge * H: y0 = 0
        if y1 >= (1 - gap_to_edge) * H - 1: y1 = H - 1
        yy = np.arange(y0, y1 + 1); xb = np.interp(yy, rows, ext)
        xb = cv2.medianBlur(xb.astype(np.float32).reshape(-1, 1), 5).ravel() if xb.size >= 5 else xb
        m = np.zeros((H, W), bool); cols = np.arange(W)[None, :]
        m[yy] = (cols <= xb[:, None]) if cw == 'left' else (cols >= xb[:, None])
        return m
    k = max(3, int(0.006 * max(H, W)) | 1); ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    zone = cv2.dilate((proxy > 0).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(0.2 * W) | 1,) * 2)) > 0
    # zone = breast found by the rule-based mask, widened by 10% of the image width on every side
    def clean(line, red_line):
        # drop marks that are not the boundary line, above all the image number written in red in the background:
        # a red piece is kept only if most of it lies in the zone around the breast; yellow pieces are kept if they are
        # at least 15% of the largest yellow piece
        n, lab, st, _ = cv2.connectedComponentsWithStats((cv2.dilate(line.astype(np.uint8), ker) > 0).astype(np.uint8), connectivity=8)
        if n <= 1: return line > 0, 0
        tot = np.bincount(lab.ravel(), minlength=n); inz = np.bincount(lab[zone], minlength=n)
        ok = inz >= 0.5 * tot if red_line else st[:, cv2.CC_STAT_AREA] >= 0.15 * st[1:, cv2.CC_STAT_AREA].max()
        ok[0] = False; keep = np.flatnonzero(ok)
        return (line > 0) & np.isin(lab, keep), int(n - 1 - len(keep))
    def main_part(m):
        # keep only the part(s) of the region that lie mostly on the breast: a red number drawn just above the breast can
        # survive the zone filter and add a separate strip in the background; such a strip barely overlaps the breast
        n, lab = cv2.connectedComponents(m.astype(np.uint8), connectivity=8)
        if n <= 2: return m, 0
        tot = np.bincount(lab.ravel(), minlength=n); inb = np.bincount(lab[proxy > 0], minlength=n)
        ok = inb >= 0.5 * tot; ok[0] = False
        if not ok.any(): ok[np.argmax(np.r_[0, tot[1:]])] = True
        return np.isin(lab, np.flatnonzero(ok)), int(n - 1 - ok.sum())
    def flood(line):                                    # region enclosed by the line and the chest-wall side (valid only if the line is closed)
        wall = cv2.dilate(line.astype(np.uint8), ker) > 0
        n, lab = cv2.connectedComponents((~wall).astype(np.uint8), connectivity=4)
        bg = set(np.unique(lab[:, [W - 1] if cw == 'left' else [0]])) - {0}
        reg = ~np.isin(lab, list(bg)); reg &= ~(wall & (cv2.dilate(np.isin(lab, list(bg)).astype(np.uint8), ker) > 0))
        n2, lab2 = cv2.connectedComponents(reg.astype(np.uint8), connectivity=4)        # drop enclosed holes (e.g. inside digits)
        side = set(np.unique(lab2[:, [0] if cw == 'left' else [W - 1]])) - {0}
        return np.isin(lab2, list(side))
    red, n_drop_red = clean(red, True); yel, n_drop_yel = clean(yel, False) if yel.sum() > 50 else (yel > 0, 0)
    rb = row_boundary(red)
    if rb is None: rb = np.zeros((H, W), bool)
    fl = flood(red)
    # a closed line gives the exact enclosed region; if the line has a gap the fill leaks and leaves only a thin strip,
    # so the closed-line region is used only when it covers at least a quarter of the rule-based breast area
    method = 'closed line' if fl.sum() >= 0.25 * max((proxy > 0).sum(), 1) else 'row boundary'
    breast_full, n_strip = main_part(fl if method == 'closed line' else rb)
    pect = np.zeros_like(breast_full)
    if yel.sum() > 50:
        pm = row_boundary(yel, gap_to_edge=0.15)          # pectoral line runs from the top edge down to the chest-wall edge
        if pm is not None:
            ys = np.nonzero(yel)[0]; pm[ys.max() + 1:] = False
            pect = pm & breast_full
    return dict(full=breast_full, pect=pect, cw=cw, proxy=proxy > 0, gray=gray, red=red, yel=yel, method=method, frac_row=rb.mean(), frac_flood=fl.mean(), dropped=n_drop_red + n_drop_yel, strips=n_strip)
R2 = {f: regions(f) for f in sub.original_filename}
for f, r in R2.items():
    print(f'{f:28s} chest wall {r["cw"]:5s} | breast {r["full"].mean():.3f} | pectoral {r["pect"].mean():.3f} | marks removed {r["dropped"]} | strips removed {r["strips"]} | {r["method"]:12s} (row {r["frac_row"]:.3f}, closed {r["frac_flood"]:.3f}) | view {sub.set_index("original_filename").loc[f, "view"]}')

In [ ]:
# 8 · Write reader 2 masks at 256 x 256 and check agreement with the stored P1-25 values
ir_old = pd.read_csv(f'{OUT}/annot_results/reader2/inter_reader.csv').set_index('file_id')
chk = []
for _, row in sub.iterrows():
    r = R2[row.original_filename]; side = 'left' if r['cw'] == 'right' else 'right'
    f2 = to256(r['full'], side); b2 = to256(r['full'] & ~r['pect'], side)
    cv2.imwrite(f'{REFS}/reader2/full/{row.file_id}.png', f2.astype(np.uint8) * 255)
    cv2.imwrite(f'{REFS}/reader2/breast/{row.file_id}.png', b2.astype(np.uint8) * 255)
    f1 = cv2.imread(f'{REFS}/reader1/full/{row.file_id}.png', 0) > 127
    d = 2 * (f1 & f2).sum() / (f1.sum() + f2.sum()); chk.append(abs(d - ir_old.loc[row.file_id, 'inter_full']))
print('reader 2 masks written:', len(sub), '| max |inter-reader Dice - stored value|:', round(max(chk), 6))
print('DONE')